# Modul A · Kapitel 2 · Teil 3 — Feed-Forward und Norm Layer

## Challenge: Feed-Forward, Skip Connection, Norm Layer, das ganze Modell

**Lernziel:** Du kannst erklären, warum auf die Aufmerksamkeit ein Feed-Forward-Netz folgt und
wozu Skip Connections und Norm Layer da sind — und du setzt aus den Bausteinen ein vollständiges
Sprachmodell zusammen und rechnest nach, wo seine Parameter stecken.

```
Eingabe
   │
   ▼
Tokenizer ────────────► Word Embedding ──┐
   │                                     ├──► Add ──┐
   └──────────────────► Positional       │          │
                        Encoding ────────┘          │
                                                    ▼
   ┌────────────────────────────────────────────────────────────┐
   │  Skip ────────────────────────────────────────┐            │
   │  Norm ──► Multi-Head Attention ──► Add ◄──────┘            │
   │                                     │                      │
   │  Skip ────────────────────────────────────────┐            │
   │  Norm ──► Feed-Forward ──────────► Add ◄──────┘            │
   └───────────────────── × N_LAYER ─────┼──────────────────────┘
                                         ▼
             Abschluss-Norm ──► Lineare Schicht ──► Softmax
                                         │
                                         ▼
                 Wahrscheinlichkeit für das nächste Zeichen
```

Der Kasten in der Mitte und alles, was darunter steht, entsteht in diesem Notebook. Am Ende
läuft ein Textstück durch das komplette Modell, und heraus kommen 87 Wahrscheinlichkeiten — eine
je möglichem nächsten Zeichen.

Jeden Baustein bauen wir erst **so einfach wie möglich selbst** und schauen danach, was **echte
Modelle** an derselben Stelle tun.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |
| 💬 | Diskussionsfrage — kurz überlegen, gerne mit der Nachbarin / dem Nachbarn |

**Wichtig:** Führe die Zellen **von oben nach unten** aus. Spätere Zellen brauchen die
Klassen, die du vorher schreibst.

Es sind **4 Challenges**.

---
## 0 · Setup

▶️ Führe diese drei Zellen aus. Sie bringen alles mit, was der Block als Eingabe braucht:
Tokenizer und Vokabular des Faust, die Einstellungen des Modells, `hole_batch`, die beiden
Embedding-Tabellen und die fertigen Klassen `Kopf` und `MehrereKoepfe`.

Das Notebook läuft damit eigenständig — es setzt kein anderes voraus.

In [ ]:
# ▶️ Bibliotheken, Farben, Zufallsstartwert, Gerät
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.nn import functional as F

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 11,
})
torch.manual_seed(1337)

# Wenn eine Grafikkarte da ist, benutzen wir sie. Gerechnet wird in diesem Notebook wenig,
# der Unterschied fällt also kaum auf.
if torch.cuda.is_available():
    GERAET = "cuda"                                  # NVIDIA-Grafikkarte, z. B. in Colab
elif torch.backends.mps.is_available():
    GERAET = "mps"                                   # Apple Silicon
else:
    GERAET = "cpu"


def zahl(n):
    """Große Zahlen mit Tausenderpunkten, wie im Deutschen üblich."""
    return f"{n:,}".replace(",", ".")


print(f"PyTorch {torch.__version__}")
print(f"Gerät:   {GERAET}")
print("Setup fertig ✔")

In [ ]:
# ▶️ Text, Vokabular, Tokenizer, Einstellungen, Trainingsbeispiele
def lade_faust():
    """Lädt Goethes Faust I und II — einmalig von Project Gutenberg, danach aus `faust.txt`."""
    import re
    import urllib.request

    pfad = Path("faust.txt")
    if pfad.exists():
        print(f"Gelesen: {pfad}")
        return pfad.read_text(encoding="utf-8")

    print("faust.txt nicht gefunden — lade von Project Gutenberg (rund 0,5 MB) …")

    def teil(nummer, ab):
        """Holt ein Buch und schneidet Vorspann, Nachspann und Inhaltsverzeichnis weg."""
        adresse = f"https://www.gutenberg.org/cache/epub/{nummer}/pg{nummer}.txt"
        roh = urllib.request.urlopen(adresse).read().decode("utf-8").replace("\r\n", "\n")
        roh = roh[:roh.find("*** END OF THE PROJECT GUTENBERG")]
        roh = "\n".join(z[2:] if z.startswith("  ") else z for z in roh.split("\n"))
        return roh[roh.find(ab):]

    erster = teil(2229, "Zueignung\n\n\nIhr naht")
    zweiter = teil(2230, "1.  Akt--Anmutige Gegend")
    # Faust II schreibt die Sprechernamen mit Doppelpunkt, Faust I mit Punkt — wir vereinheitlichen
    zweiter = re.sub(r"(?m)^([A-ZÄÖÜ][A-ZÄÖÜ \-]*(?:\(.*?\))?):$", r"\1.", zweiter)

    text = re.sub(r"\n{3,}", "\n\n", erster.strip() + "\n\n" + zweiter.strip()) + "\n"
    Path("faust.txt").write_text(text, encoding="utf-8")
    print("Gespeichert als: faust.txt")
    return text


text = lade_faust()

# Ein Token ist ein Zeichen
zeichen = sorted(set(text))
vokabular_groesse = len(zeichen)
zeichen_zu_zahl = {z: i for i, z in enumerate(zeichen)}
zahl_zu_zeichen = {i: z for i, z in enumerate(zeichen)}


def kodiere(s):
    """Text → Liste von Zahlen."""
    return [zeichen_zu_zahl[z] for z in s]


def dekodiere(zahlen):
    """Liste von Zahlen → Text."""
    return "".join(zahl_zu_zeichen[i] for i in zahlen)


# Die Einstellungen des Modells
KONTEXT = 96      # wie viele Zeichen das Modell zurückschaut
BATCH = 32        # wie viele Textstücke gleichzeitig gerechnet werden
N_EMBD = 96       # Länge der Vektoren, mit denen das Modell intern arbeitet
N_HEAD = 4        # Aufmerksamkeitsköpfe je Block
N_LAYER = 3       # wie viele Blöcke übereinander
DROPOUT = 0.1     # Anteil der Verbindungen, der beim Training zufällig abgeschaltet wird

# 90 % Training, 10 % Validierung
daten = torch.tensor(kodiere(text), dtype=torch.long)
grenze = int(0.9 * len(daten))
train_daten, val_daten = daten[:grenze], daten[grenze:]


def hole_batch(art):
    """Zieht BATCH zufällige Textstücke. art ist "train" oder "val"."""
    quelle = train_daten if art == "train" else val_daten
    start = torch.randint(len(quelle) - KONTEXT - 1, (BATCH,))
    x = torch.stack([quelle[i:i + KONTEXT] for i in start])
    y = torch.stack([quelle[i + 1:i + KONTEXT + 1] for i in start])
    return x.to(GERAET), y.to(GERAET)


print(f"{zahl(len(text))} Zeichen, {vokabular_groesse} verschiedene")
print(f"{N_LAYER} Blöcke à {N_HEAD} Köpfe, {N_EMBD} Dimensionen, Kontextlänge {KONTEXT}")
print(f"Ein Batch: {tuple(hole_batch('train')[0].shape)}")

In [ ]:
# ▶️ Embeddings und Aufmerksamkeit — fertig, damit der Block eine echte Eingabe bekommt
token_embedding = nn.Embedding(vokabular_groesse, N_EMBD)      # was steht da?
positions_embedding = nn.Embedding(KONTEXT, N_EMBD)            # an welcher Stelle?


def einbetten(idx):
    """(B, T) Token-Nummern → (B, T, N_EMBD): Token-Vektor plus Positions-Vektor."""
    T = idx.shape[1]
    return token_embedding(idx) + positions_embedding(torch.arange(T, device=idx.device))


class Kopf(nn.Module):
    """Ein einzelner Aufmerksamkeitskopf: Query, Key, Value, maskiert, Softmax."""

    def __init__(self, kopf_groesse):
        super().__init__()
        self.query = nn.Linear(N_EMBD, kopf_groesse, bias=False)
        self.key = nn.Linear(N_EMBD, kopf_groesse, bias=False)
        self.value = nn.Linear(N_EMBD, kopf_groesse, bias=False)
        self.dropout = nn.Dropout(DROPOUT)
        self.register_buffer("maske", torch.tril(torch.ones(KONTEXT, KONTEXT)))

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        gewichte = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        gewichte = gewichte.masked_fill(self.maske[:T, :T] == 0, float("-inf"))
        gewichte = F.softmax(gewichte, dim=-1)
        gewichte = self.dropout(gewichte)
        return gewichte @ v


class MehrereKoepfe(nn.Module):
    """N_HEAD Aufmerksamkeitsköpfe parallel, danach wieder zusammengemischt."""

    def __init__(self, anzahl, kopf_groesse):
        super().__init__()
        self.koepfe = nn.ModuleList([Kopf(kopf_groesse) for _ in range(anzahl)])
        self.projektion = nn.Linear(anzahl * kopf_groesse, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        zusammen = torch.cat([kopf(x) for kopf in self.koepfe], dim=-1)
        return self.dropout(self.projektion(zusammen))


# Ein kurzer Textanfang als Probeeingabe für alle folgenden Bausteine
PROBE_TEXT = "Habe nun, ach! Philosophie"
probe_idx = torch.tensor([kodiere(PROBE_TEXT)])       # (1, T)
with torch.no_grad():                                 # nur Eingabe, keine Ableitungen nötig
    probe_x = einbetten(probe_idx)                    # (1, T, N_EMBD)

print(f"Probeeingabe {PROBE_TEXT!r}")
print(f"  Token-Nummern:  {tuple(probe_idx.shape)}")
print(f"  eingebettet:    {tuple(probe_x.shape)}   (Batch, Länge, Dimension)")
print(f"  Aufmerksamkeit: {tuple(MehrereKoepfe(N_HEAD, N_EMBD // N_HEAD)(probe_x).shape)}")

## 1 · Feed-Forward: Informationen weiterverarbeiten

📖 Nach der **Attention** weiß jedes Token besser, **was die anderen Tokens gesagt haben**.

Stell dir vor, jedes Token ist ein Mensch in einer Gruppe:

* Bei der **Attention** hört der Mensch den anderen zu und sammelt wichtige Informationen ein.
* Danach hat er viele neue Informationen im Kopf.
* Jetzt muss er diese Informationen noch **verarbeiten**.

Genau dafür gibt es das **Feed-Forward-Netz**, was nichts anderes ist als ein Neuronales Netz, das Informationen verarbeitet.

Es nimmt die Informationen eines Tokens und rechnet mit ihnen weiter.

| Baustein         | Aufgabe                                         |
| ---------------- | ----------------------------------------------- |
| **Attention**    | Informationen von **anderen Tokens einsammeln** |
| **Feed-Forward** | die gesammelten Informationen **verarbeiten**   |

Das Feed-Forward-Netz arbeitet dabei mit **jedem Token einzeln**. Die Tokens sprechen hier also nicht mehr miteinander.

Bei uns hat ein Token zuerst **96 Zahlen**.

Das Feed-Forward-Netz macht daraus kurz viel mehr Platz:

**96 → 384 → 96**

Man kann sich das wie einen großen Arbeitstisch vorstellen:

1. Die 96 Zahlen werden auf **384 Zahlen erweitert**.
2. Auf diesem größeren Platz kann das Netzwerk die Informationen besser verarbeiten.
3. Danach werden die 384 Zahlen wieder auf **96 Zahlen zusammengefasst**.

Zwischen den beiden Schritten benutzen wir **ReLU**.

Zur Errinerung: ReLU macht etwas sehr Einfaches:

* negative Zahl → **0**
* positive Zahl → **bleibt so**

Zum Beispiel:

`[-3, 2, -1, 5] → [0, 2, 0, 5]`

So kann das Netzwerk entscheiden, welche berechneten Informationen es weitergeben möchte.

### 🛠️ Challenge 1 — Das Feed-Forward-Netz

Baue die Schichtenfolge in `nn.Sequential`:

`Linear(N_EMBD → 4·N_EMBD)` → `ReLU` → `Linear(4·N_EMBD → N_EMBD)` → `Dropout`

*Tipp: `nn.Sequential(a, b, c)` hängt Schichten hintereinander. Eine lineare Schicht ist
`nn.Linear(eingang, ausgang)`, die Aktivierungsfunktion `nn.ReLU()`, und das Dropout am Ende
`nn.Dropout(DROPOUT)`.*

In [ ]:
class FeedForward(nn.Module):
    """Zwei lineare Schichten mit ReLU dazwischen — je Position einzeln angewendet."""

    def __init__(self):
        super().__init__()
        # TODO: die vier Schichten in der richtigen Reihenfolge
        self.netz = nn.Sequential(
            ...
        )

    def forward(self, x):
        return self.netz(x)

In [ ]:
# ✅ Selbsttest
try:
    probe_ff = FeedForward().eval()      # eval() schaltet das Dropout ab
except TypeError as fehler:
    raise AssertionError(
        "FeedForward lässt sich nicht bauen: In nn.Sequential(...) steht etwas, das keine Schicht "
        "ist. Sind die TODOs noch offen — oder fehlen Klammern (nn.ReLU statt nn.ReLU())?  "
        f"Meldung von PyTorch: {fehler}"
    ) from fehler

assert probe_ff(probe_x).shape == probe_x.shape, \
    f"Ein- und Ausgang müssen dieselbe Form haben, hier: {tuple(probe_ff(probe_x).shape)}"

erwartet = 2 * N_EMBD * 4 * N_EMBD + 4 * N_EMBD + N_EMBD
gezaehlt = sum(p.numel() for p in probe_ff.parameters())
assert gezaehlt == erwartet, \
    f"Erwartet werden {zahl(erwartet)} Parameter (96 → 384 → 96), gezählt {zahl(gezaehlt)}"

# Ist wirklich eine Aktivierungsfunktion dabei? Eine Kette aus rein linearen Schichten erfüllt
# f(a + b) - f(a) - f(b) + f(0) = 0. Bei Nichtlinearität steht dort etwas deutlich anderes.
with torch.no_grad():
    a, b = torch.randn(1, 4, N_EMBD), torch.randn(1, 4, N_EMBD)
    rest = probe_ff(a + b) - probe_ff(a) - probe_ff(b) + probe_ff(torch.zeros(1, 4, N_EMBD))
assert rest.abs().max() > 0.01, (
    "Ohne Aktivierungsfunktion sind zwei lineare Schichten hintereinander wieder eine einzige — "
    f"gemessene Abweichung von der Linearität: {rest.abs().max():.2e}"
)

print(f"✅ Challenge 1 gelöst — {zahl(gezaehlt)} Parameter je Feed-Forward")
print(f"   Abweichung von der Linearität: {rest.abs().max():.3f}   (bei 0 wäre keine da)")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
        self.netz = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD),
            nn.ReLU(),
            nn.Linear(4 * N_EMBD, N_EMBD),
            nn.Dropout(DROPOUT),
        )
```

`nn.Sequential` reicht die Eingabe durch alle Schichten der Reihe nach. Die lineare Schicht
arbeitet dabei nur auf der **letzten** Dimension: Aus `(1, 26, 96)` wird `(1, 26, 384)`, die
ersten beiden Achsen bleiben unberührt. Genau das heißt „je Position einzeln".

</details>

### Wozu die ReLU Schicht?

In [ ]:
# Die Feed-Forward-Schicht vergrößert jeden Token-Vektor:
# 96 Zahlen → 384 Zahlen
breite_schicht = nn.Linear(N_EMBD, 4 * N_EMBD)

with torch.no_grad():

    # Für jede Token-Position entstehen 384 neue Werte.
    # flatten() legt anschließend die Werte aller Positionen
    # zu einer langen Liste zusammen.
    vor = breite_schicht(probe_x).flatten()

    # ReLU lässt positive Werte unverändert
    # und setzt alle negativen Werte auf 0.
    nach = F.relu(vor)


# Wie viel Prozent der Werte wurden von ReLU auf 0 gesetzt?
anteil_null = (nach == 0).float().mean().item()


# Bereiche für das Histogramm
kaesten = np.linspace(-4, 4, 61)


# Verteilung vor und nach der ReLU vergleichen
fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4))

links.hist(vor.numpy(), bins=kaesten, color=GRAU)
links.set_title("vor der ReLU")

rechts.hist(nach.numpy(), bins=kaesten, color=BLAU)
rechts.set_title(f"nach der ReLU — {anteil_null:.0%} sind exakt null")


# Die gestrichelte Linie markiert den Wert 0
for achse in (links, rechts):
    achse.axvline(0, color=ORANGE, lw=1.5, ls="--")
    achse.set_xlabel("Wert")
    achse.set_ylabel("Anzahl")

plt.tight_layout()
plt.show()


# Anzahl aller betrachteten Werte ausgeben
print(
    f"{zahl(len(vor))} Zahlen, "
    f"davon nach der ReLU {zahl(int((nach == 0).sum()))} exakt null"
)

📖 Durch **ReLU** kann das Netzwerk bestimmte berechnete Informationen **aktivieren** und andere **ausblenden**.

Zum Beispiel:

`[-4, 3, -2, 7] → ReLU → [0, 3, 0, 7]`

Das ist wichtig, denn ohne ReLU würden die beiden linearen Schichten einfach nur hintereinander rechnen. Zwei lineare Schichten ohne Aktivierungsfunktion könnte man zu **einer einzigen linearen Schicht zusammenfassen**. Erst durch ReLU kann das Netzwerk lernen: Dadurch wird aus einfachem Addieren eine echte **Entscheidung darüber, welche Muster aktiviert werden**.

### Was macht Dropout?

In [ ]:
# ▶️ Dropout schaltet beim Training zufällig einige Werte aus

dropout = nn.Dropout(p=0.3)

# Beispiel: 20 positive Aktivierungen
werte = torch.ones(20)

# Dropout ist nur im Trainingsmodus aktiv
dropout.train()

with torch.no_grad():
    nach = dropout(werte)

print("vorher: ", werte.tolist())
print("nachher:", nach.tolist())

anteil_null = (nach == 0).float().mean().item()
print(f"{anteil_null:.0%} der Werte wurden ausgeschaltet")

📖 Dropout verhindert, dass sich das Netzwerk zu sehr auf einzelne Aktivierungen verlässt.

Während des Trainings werden zufällig einige Werte auf 0 gesetzt. Bei p=0.3 werden beispielsweise ungefähr 30 % der Aktivierungen ausgeschaltet. Das Netzwerk muss deshalb lernen, auch ohne diese Werte zu einer guten Vorhersage zu kommen.

Die übrigen Aktivierungen werden automatisch etwas verstärkt, damit ihre durchschnittliche Größenordnung gleich bleibt. Deswegen wird aus `1.0` → `1.428...`.

Wichtig: Dropout ist nur während des Trainings aktiv. Bei der späteren Vorhersage werden keine Werte mehr zufällig ausgeschaltet.

## 2 · Skip Connection und Norm Layer

📖 Ein Transformer besteht aus vielen Schichten hintereinander. Je tiefer das Netzwerk wird, desto schwieriger wird das Training.

Deshalb gibt es zwei kleine Helfer:

* **Skip Connection** → baut eine Abkürzung
* **Norm Layer** → hält die Zahlen in einem gut handhabbaren Bereich

Beide haben nichts speziell mit Sprache zu tun. Sie helfen einfach dabei, dass ein tiefes neuronales Netz stabil lernen kann.

### 2.1 · Skip Connection: eine Abkürzung durch das Netzwerk

Normalerweise könnte eine Schicht einfach die alten Informationen ersetzen:

```python
x = baustein(x)
```

Mit einer **Skip Connection** behalten wir die alten Informationen zusätzlich:

```python
x = x + baustein(x)
```

Der Baustein sagt also nicht:

**„Hier ist eine komplett neue Version von `x`.“**

Sondern eher:

**„Behalte dein altes `x` und ergänze nur das, was ich gelernt habe.“**

Das ist besonders beim Training wichtig.

Beim Lernen wird ein **Gradient** von hinten durch das Netzwerk zurückgeschickt. Er sagt den früheren Schichten ungefähr:

**„So musst du dich verändern, damit der Fehler kleiner wird.“**

Bei sehr vielen Schichten kann dieses Lernsignal unterwegs immer kleiner werden. Dann kommt bei den ersten Schichten fast nichts mehr an und sie lernen kaum noch.

Die Skip Connection baut deshalb eine **Abkürzung**:

```text
x ───────────────────────► +
│                          │
└──► Baustein(x) ────────► +
```

Der Gradient kann jetzt nicht nur durch den komplizierten Baustein zurücklaufen, sondern hat zusätzlich einen **direkten Weg über `x`**.

Dadurch erreicht das Lernsignal auch frühere Schichten viel leichter.

Kurz gesagt:

**Skip Connections helfen der Information nach vorne und dem Lernsignal nach hinten.**

▶️ Das können wir direkt messen: Wir bauen zweimal 24 Schichten hintereinander — einmal **mit** und einmal **ohne Skip Connections** — und schauen danach, wie viel Gradient noch bei den einzelnen Schichten ankommt.

In [ ]:
# ▶️ Gradientennorm je Schicht, mit und ohne Skip Connection
TIEFE = 24


def gradienten(mit_skip):
    """Baut einen Stapel aus TIEFE Schichten, macht einen Rückwärtsdurchlauf und
    gibt die Gradientennorm je Schicht zurück (unten → oben)."""
    torch.manual_seed(0)
    schichten = nn.ModuleList([
        nn.Sequential(nn.Linear(N_EMBD, N_EMBD), nn.ReLU()) for _ in range(TIEFE)
    ])

    h = torch.randn(64, N_EMBD)
    for schicht in schichten:
        h = h + schicht(h) if mit_skip else schicht(h)

    h.pow(2).mean().backward()
    return [s[0].weight.grad.norm().item() for s in schichten]


ohne, mit = gradienten(mit_skip=False), gradienten(mit_skip=True)

plt.figure(figsize=(8, 4.5))
plt.semilogy(range(1, TIEFE + 1), ohne, color=GRAU, lw=2, marker="o", ms=4,
             label="ohne Skip Connection")
plt.semilogy(range(1, TIEFE + 1), mit, color=BLAU, lw=2, marker="o", ms=4,
             label="mit Skip Connection")
plt.xlabel("Schicht (1 = unterste, 24 = oberste)")
plt.ylabel("Größe des Gradienten")
plt.title("Wie viel Gradient kommt unten an?")
plt.legend()
plt.show()

for name, werte in [("ohne Skip", ohne), ("mit Skip", mit)]:
    print(f"{name:<12} unterste Schicht {werte[0]:.2e}   oberste {werte[-1]:.2e}   "
          f"Verhältnis {werte[0] / werte[-1]:.2e}")

📖 Die Achse ist logarithmisch, jeder Strich ein Faktor 10. Zu vergleichen ist der **Verlauf**
jeder Kurve von rechts nach links, nicht die Höhe der beiden zueinander — die hängt daran, wie
groß der Ausgang des jeweiligen Stapels insgesamt ist.

Ohne Skip Connection fällt der Gradient über 24 Schichten um rund **neun Zehnerpotenzen** ab: In
der untersten Schicht kommt etwa ein Milliardstel von dem an, was oben anliegt. Ihre Parameter
würden sich praktisch nicht mehr bewegen.

Mit Skip Connection bleiben unten und oben in derselben Größenordnung — Faktor 4 statt Faktor
1.000.000.000. Genau deshalb sind 24, 96 oder mehr Schichten übereinander überhaupt möglich.

Der Effekt hat einen Namen: **verschwindende Gradienten**. Die Skip Connection ist die Lösung,
die sich durchgesetzt hat.

### 2.2 · Norm Layer: die Zahlen auf einer vernünftigen Größe halten

📖 Während ein Vektor durch viele Schichten läuft, können seine Zahlen immer größer oder immer kleiner werden.

Zum Beispiel könnte ein Vektor irgendwann so aussehen:

`[120, -80, 250, ...]`

und ein anderer eher so:

`[0.02, -0.01, 0.03, ...]`

Für das Netzwerk ist das beim Lernen ungünstig. Wenn die Größen ständig stark schwanken, wird das Training schwieriger und instabiler.

Deshalb gibt es **Layer Normalization**.

Sie nimmt jeden einzelnen Vektor — bei uns also die **96 Zahlen eines Tokens** — und bringt seine Zahlen wieder auf eine ähnliche Größenordnung.

Vereinfacht passiert dabei:

1. Wir schauen: **Wo liegt die Mitte der 96 Zahlen?**
2. Wir verschieben die Zahlen so, dass ihre Mitte bei **0** liegt.
3. Wir skalieren sie so, dass sie eine **gut kontrollierte Streuung** haben.

Mathematisch:

$$
\hat{x} =
\frac{x-\text{Mittelwert}(x)}
{\sqrt{\text{Varianz}(x)+\varepsilon}}
$$

Danach hat der Vektor ungefähr:

* **Mittelwert = 0**
* **Standardabweichung = 1**

Wichtig: Jeder Token-Vektor wird **für sich selbst** normalisiert.

Wenn unser Satz also 10 Tokens enthält, wird nicht über alle Tokens gemeinsam gerechnet. Für jedes Token werden nur seine eigenen **96 Zahlen** betrachtet.

```text
Token 1: [96 Zahlen] → normalisieren
Token 2: [96 Zahlen] → normalisieren
Token 3: [96 Zahlen] → normalisieren
...
```

So bleiben die Zahlen auch nach vielen Transformer-Schichten in einem Bereich, mit dem das Netzwerk gut weiterarbeiten und lernen kann.

Das kleine $\varepsilon$ ist nur eine Sicherheitsmaßnahme: Es verhindert, dass wir versehentlich durch **0** teilen.

### 🛠️ Challenge 2 — Layer Normalization selbst rechnen

`normalisiere(x)` bekommt einen Tensor der Form `(B, T, C)` und normalisiert **jeden Vektor
entlang der letzten Achse**: Mittelwert 0, Standardabweichung 1. Die Form bleibt gleich.

*Tipp: `x.mean(dim=-1, keepdim=True)` mittelt über die letzte Achse und behält die Achse als
Länge 1, damit man direkt subtrahieren kann. Für die Streuung nimm
`x.var(dim=-1, keepdim=True, unbiased=False)` — `unbiased=False` teilt durch 96 statt durch 95,
und genau so rechnet auch PyTorch. Die Wurzel zieht `torch.sqrt(...)`.*

In [ ]:
EPSILON = 1e-5


def normalisiere(x):
    """(B, T, C) → (B, T, C): jeder Vektor mit Mittelwert 0 und Standardabweichung 1."""
    # TODO 1: Mittelwert über die letzte Achse
    mittel = ...

    # TODO 2: Varianz über die letzte Achse (unbiased=False)
    streuung = ...

    # TODO 3: zentrieren und durch die Wurzel aus (Varianz + EPSILON) teilen
    return ...

In [ ]:
# ✅ Selbsttest
schief = probe_x * 5 + 3                 # absichtlich verschoben und gestreckt
eigen = normalisiere(schief)

assert isinstance(eigen, torch.Tensor), \
    "normalisiere() gibt keinen Tensor zurück — die TODOs sind noch offen"
assert eigen.shape == schief.shape, f"Die Form muss erhalten bleiben, ist {tuple(eigen.shape)}"
assert eigen.mean(dim=-1).abs().max() < 1e-4, \
    f"Der Mittelwert je Vektor muss 0 sein, größter gemessener Wert: {eigen.mean(-1).abs().max():.3f}"
assert (eigen.std(dim=-1, unbiased=False) - 1).abs().max() < 1e-3, \
    "Die Standardabweichung je Vektor muss 1 sein — mit unbiased=False rechnen"
assert torch.allclose(eigen, nn.LayerNorm(N_EMBD)(schief), atol=1e-4), \
    "Das Ergebnis muss mit nn.LayerNorm übereinstimmen"

# Je Vektor heißt: nicht über den Batch. Ein zweiter Eintrag im Batch ändert am ersten nichts.
zwei = torch.cat([schief, schief * 100], dim=0)
assert torch.allclose(normalisiere(zwei)[0], eigen[0], atol=1e-5), \
    "Normalisiert wird je Vektor — was sonst im Batch liegt, darf keine Rolle spielen"

print("✅ Challenge 2 gelöst")
print()
print(f"vor  der Normalisierung:  Mittelwert {schief.mean():.2f}   Standardabweichung {schief.std():.2f}")
print(f"nach der Normalisierung:  Mittelwert {eigen.mean():.2f}   Standardabweichung {eigen.std():.2f}")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
EPSILON = 1e-5


def normalisiere(x):
    """(B, T, C) → (B, T, C): jeder Vektor mit Mittelwert 0 und Standardabweichung 1."""
    mittel = x.mean(dim=-1, keepdim=True)

    streuung = x.var(dim=-1, keepdim=True, unbiased=False)

    return (x - mittel) / torch.sqrt(streuung + EPSILON)
```

`nn.LayerNorm(96)` macht genau diese Rechnung und hat zusätzlich 96 + 96 = **192 Parameter**:
einen Faktor und einen Summanden je Dimension. Damit kann das Modell die Normalisierung
nachträglich wieder verschieben und strecken, wenn ihm das nützt. Es fängt bei Faktor 1 und
Summand 0 an — also bei genau deiner Funktion.

</details>

In [ ]:
# ▶️ Mittelwert und Standardabweichung je Position, vor und nach der Normalisierung
norm = nn.LayerNorm(N_EMBD)

with torch.no_grad():
    danach = norm(schief)

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

for achse, daten_stueck, titel in [(links, schief, "vor der Normalisierung"),
                                   (rechts, danach, "nach der Normalisierung")]:
    achse.plot(daten_stueck[0].mean(-1), color=BLAU, lw=2, marker="o", ms=4, label="Mittelwert")
    achse.plot(daten_stueck[0].std(-1), color=ORANGE, lw=2, marker="o", ms=4,
               label="Standardabweichung")
    achse.axhline(0, color=GRAU, lw=1)
    achse.axhline(1, color=GRAU, lw=1, ls=":")
    achse.set_title(titel)
    achse.set_xlabel("Position im Text")
    achse.legend()

links.set_ylabel("Wert über die 96 Dimensionen")
plt.tight_layout()
plt.show()

📖 Rechts liegen alle Mittelwerte exakt auf 0 und alle Standardabweichungen exakt auf 1 — Position
für Position. Links schwanken beide.

Wichtig ist der Unterschied zur **Batch Normalization**, die man aus Bildnetzen kennt: Die
normalisiert über die Beispiele eines Batches. Beim Erzeugen von Text besteht der Batch aber oft
aus einem einzigen Beispiel, und Textstücke sind unterschiedlich lang. Layer Normalization hat
dieses Problem nicht, weil ihre Statistik allein aus dem einen Vektor kommt.

💬 **`nn.LayerNorm` hat 192 lernbare Parameter — Faktor und Summand je Dimension. Warum
normalisiert man erst auf Mittelwert 0 und Standardabweichung 1, wenn das Modell beides
anschließend wieder verändern darf?**

<details>
<summary>Antwort aufklappen</summary>

Weil sich dadurch ändert, **wovon** die Größenordnung abhängt. Ohne Normalisierung ergibt sie
sich aus allem, was vorher passiert ist: aus den Eingabedaten, aus 20 Schichten davor, aus dem
aktuellen Trainingsstand. Sie schwankt ständig und unkontrolliert.

Nach der Normalisierung steht die Größenordnung an dieser Stelle in zwei Parametern, die direkt
gelernt werden. Das Modell kann sie einstellen — es muss sie aber nicht mühsam über viele
Schichten hinweg ausbalancieren.

In der Praxis bleiben die gelernten Faktoren meist nahe bei 1. Der Nutzen liegt nicht darin,
dass sie sich stark verändern, sondern darin, dass die Werte an dieser Stelle nie entgleisen.

</details>

---
### 2.3 · Pre-Norm oder Post-Norm

📖 Bleibt die Frage, **wo** die Normalisierung steht. Es gibt zwei Anordnungen:

| | Rechnung | benutzt von |
|---|---|---|
| **Post-Norm** | `x = norm(x + baustein(x))` | Transformer-Aufsatz 2017 |
| **Pre-Norm** | `x = x + baustein(norm(x))` | GPT-2, GPT-3, Llama, alle modernen Modelle |

Die übliche Abbildung des Transformers zeigt **Post-Norm** — erst addieren, dann normalisieren.
So stand es 2017 im Aufsatz, und so ist es in fast jeder Nachzeichnung seither gezeichnet.
Begegnet dir das Bild mit der Norm hinter dem Addieren, ist es die ältere Anordnung.

**Wir bauen Pre-Norm**, wie GPT-2 und alles danach; das Flussdiagramm ganz oben in diesem
Notebook ist deshalb schon so gezeichnet. Der Grund lässt sich exakt vorführen: Bei Post-Norm
läuft auch der direkte Weg der Skip Connection durch die Normalisierung. Bei Pre-Norm nicht —
dort gibt es einen Pfad vom Eingang zum Ausgang, der durch keine einzige Rechnung läuft.

▶️ Der Nachweis: Wir setzen alle Parameter der Bausteine auf null. Dann geben sie nichts mehr aus,
und übrig bleibt nur der direkte Weg.

In [ ]:
# ▶️ Was passiert, wenn die Bausteine nichts beitragen?
def leerer_stapel(x, art, tiefe=6):
    """Ein Stapel aus `tiefe` Schichten, deren Bausteine alle exakt null ausgeben."""
    schichten = nn.ModuleList([nn.Linear(N_EMBD, N_EMBD) for _ in range(tiefe)])
    normen = nn.ModuleList([nn.LayerNorm(N_EMBD) for _ in range(tiefe)])
    for schicht in schichten:
        nn.init.zeros_(schicht.weight)
        nn.init.zeros_(schicht.bias)

    h = x
    for schicht, n in zip(schichten, normen):
        h = h + schicht(n(h)) if art == "pre" else n(h + schicht(h))
    return h


torch.manual_seed(0)
eingang = torch.randn(4, N_EMBD) * 3 + 1        # bewusst nicht zentriert

with torch.no_grad():
    for art in ("pre", "post"):
        ausgang = leerer_stapel(eingang, art)
        gleich = torch.allclose(eingang, ausgang, atol=1e-5)
        print(f"{art + '-Norm':<12} Eingabe unverändert durchgereicht? {str(gleich):<6}"
              f"größte Abweichung: {(eingang - ausgang).abs().max():.3f}")

📖 Bei Pre-Norm kommt die Eingabe unverändert unten an: Der Stapel reicht sie durch, weil die
Skip Connections eine ununterbrochene Kette aus reinen Additionen bilden. Bei Post-Norm nicht —
dort greift in jeder Schicht die Normalisierung in den direkten Weg ein und verändert ihn, selbst
wenn der Baustein exakt nichts beiträgt.

Praktisch heißt das: Post-Norm-Modelle brauchen ab einer gewissen Tiefe eine vorsichtig
ansteigende Lernrate am Trainingsanfang (*Warmup*), sonst laufen sie aus dem Ruder. Pre-Norm ist
in dieser Hinsicht anspruchsloser. Deshalb hat die Anordnung ab GPT-2 gewechselt, während die
Abbildungen die alte behalten haben.

---
## 3 · Der Transformer Block

📖 Jetzt passt alles zusammen. Ein **Transformer-Block** besteht aus zwei Teilschritten, und jeder
der beiden bekommt seine eigene Normalisierung und seine eigene Skip Connection:

```
x = x + aufmerksamkeit(norm1(x))
x = x + feedforward(norm2(x))
```

Mehr ist es nicht. Die Eingabe hat die Form `(B, T, 96)`, die Ausgabe hat dieselbe Form — deshalb
lassen sich Blöcke direkt hintereinanderschalten.

### 🛠️ Challenge 3 — Der Transformer-Block

Der Aufbau steht schon da. Du schreibst `forward` — zwei Zeilen in **Pre-Norm**-Anordnung:

1. Normalisieren, durch die Aufmerksamkeit, Ergebnis **zu `x` addieren**
2. Normalisieren, durch das Feed-Forward, Ergebnis **zu `x` addieren**

*Tipp: Die Reihenfolge ist `x + baustein(norm(x))` — erst `self.norm1(x)`, das Ergebnis in
`self.aufmerksamkeit(...)`, und diese Summe zu `x` dazu. Die zweite Zeile ist dieselbe Form mit
`self.norm2` und `self.feedforward`.*

In [ ]:
class Block(nn.Module):
    """Ein Transformer-Block: Aufmerksamkeit und Feed-Forward, je mit Norm und Skip Connection."""

    def __init__(self):
        super().__init__()
        self.aufmerksamkeit = MehrereKoepfe(N_HEAD, N_EMBD // N_HEAD)
        self.feedforward = FeedForward()
        self.norm1 = nn.LayerNorm(N_EMBD)
        self.norm2 = nn.LayerNorm(N_EMBD)

    def forward(self, x):
        # TODO 1: Aufmerksamkeit mit Norm davor und Skip Connection darum
        x = ...

        # TODO 2: Feed-Forward mit Norm davor und Skip Connection darum
        x = ...

        return x

In [ ]:
# ✅ Selbsttest
probe_block = Block().eval()

assert isinstance(probe_block(probe_x), torch.Tensor), \
    "Der Block gibt keinen Tensor zurück — die TODOs in forward() sind noch offen"
assert probe_block(probe_x).shape == probe_x.shape, \
    f"Ein- und Ausgang müssen dieselbe Form haben, hier: {tuple(probe_block(probe_x).shape)}"

erwartet = (N_HEAD * 3 * N_EMBD * (N_EMBD // N_HEAD) + N_EMBD * N_EMBD + N_EMBD   # Aufmerksamkeit
            + 2 * N_EMBD * 4 * N_EMBD + 4 * N_EMBD + N_EMBD                      # Feed-Forward
            + 4 * N_EMBD)                                                        # zwei LayerNorm
gezaehlt = sum(p.numel() for p in probe_block.parameters())
assert gezaehlt == erwartet, \
    f"Erwartet werden {zahl(erwartet)} Parameter, gezählt {zahl(gezaehlt)}"

# Skip Connection: Wenn beide Bausteine exakt null ausgeben, muss der Block durchreichen.
leer = copy.deepcopy(probe_block)
with torch.no_grad():
    for p in leer.aufmerksamkeit.parameters():
        p.zero_()
    for p in leer.feedforward.parameters():
        p.zero_()
assert torch.allclose(leer(probe_x), probe_x, atol=1e-5), \
    "Ohne Beitrag der Bausteine muss die Eingabe unverändert herauskommen — die Skip Connection fehlt"

# Die Maske bleibt wirksam: Eine Änderung am Ende darf nichts davor verändern.
spaeter = probe_x.clone()
spaeter[:, -1] = torch.randn(N_EMBD)
assert torch.allclose(probe_block(probe_x)[:, :-1], probe_block(spaeter)[:, :-1], atol=1e-5), \
    "Spätere Positionen dürfen frühere Ausgaben nicht verändern"

print(f"✅ Challenge 3 gelöst — ein Block hat {zahl(gezaehlt)} Parameter")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
    def forward(self, x):
        x = x + self.aufmerksamkeit(self.norm1(x))

        x = x + self.feedforward(self.norm2(x))

        return x
```

Zwei Normalisierungen je Block, nicht eine: Aufmerksamkeit und Feed-Forward sind zwei getrennte
Teilschritte, und jeder bekommt seine Eingabe frisch normalisiert.

`x` heißt in beiden Zeilen dasselbe wie vorher plus dem Beitrag des Bausteins. Diese fortlaufend
ergänzte Größe nennt man den **Residual Stream** — sie geht durch das ganze Modell und wird von
jedem Baustein ein Stück verändert.

</details>

📖 Blöcke werden **gestapelt**. Jeder arbeitet auf dem Ergebnis des vorherigen: Die
Aufmerksamkeit im zweiten Block schaut auf Vektoren, in denen schon steht, was der erste Block
zusammengetragen hat. Damit lassen sich Zusammenhänge über Zusammenhänge ausdrücken.

Bei uns sind es 3 Blöcke, bei GPT-2 klein 12, bei GPT-3 96.

▶️ Drei Blöcke hintereinander:

In [ ]:
# ▶️ N_LAYER Blöcke stapeln
bloecke = nn.Sequential(*[Block() for _ in range(N_LAYER)]).eval()

print(f"{N_LAYER} Blöcke, {zahl(sum(p.numel() for p in bloecke.parameters()))} Parameter")
print(f"Eingabe {tuple(probe_x.shape)}  →  Ausgabe {tuple(bloecke(probe_x).shape)}   (unverändert)")
print()

# Wie stark verändert jeder einzelne Block die Darstellung?
h = probe_x
with torch.no_grad():
    for nummer, block in enumerate(bloecke, start=1):
        aus = block(h)
        aehnlichkeit = F.cosine_similarity(h, aus, dim=-1).mean().item()
        print(f"  Block {nummer}: Ähnlichkeit zwischen Ein- und Ausgang {aehnlichkeit:.3f}"
              .replace(".", ",", 1))
        h = aus

📖 Jeder Block verändert die Vektoren nur ein Stück — die Ähnlichkeit zwischen Ein- und Ausgang
liegt bei rund 0,98. Das ist die Skip Connection in Zahlen: Der Block schreibt seinen Beitrag zur
bestehenden Darstellung dazu, statt sie zu überschreiben. Über viele Blöcke summieren sich diese
Beiträge.

💬 **Was bringt es, drei Blöcke zu stapeln, statt einen einzigen Block dreimal so breit zu
machen?**

<details>
<summary>Antwort aufklappen</summary>

Ein breiterer Block rechnet **mehr** auf derselben Stufe. Gestapelte Blöcke rechnen
**aufeinander auf**.

Konkret an der Aufmerksamkeit: In einem Block kann jede Position einmal auf die Positionen davor
schauen. Mehr geht nicht — die Gewichte werden aus der Eingabe des Blocks berechnet. Im zweiten
Block schaut sie erneut, diesmal aber auf Vektoren, die bereits Information aus dem ersten
Durchgang enthalten. So werden Beziehungen zwischen Beziehungen darstellbar.

Praktisch ist es eine Abwägung: Tiefe Netze sind schwerer zu trainieren (siehe die
Gradientenmessung oben), breite Schichten lassen sich besser parallel rechnen. Große Modelle
erhöhen beides, die Tiefe aber deutlich stärker — von 12 Blöcken bei GPT-2 klein auf 96 bei
GPT-3.

</details>

---
## 4 · Die Ausgabeschicht: Welches Zeichen kommt als Nächstes?

📖 Nach dem letzten Transformer-Block hat jede Position einen Vektor mit **96 Zahlen**.

Diese 96 Zahlen enthalten alles, was das Modell an dieser Stelle über den bisherigen Text herausgearbeitet hat.

Jetzt brauchen wir aber keine 96 Zahlen mehr, sondern eine Antwort auf die Frage:

**„Welches Zeichen kommt wahrscheinlich als Nächstes?“**

In unserem Vokabular gibt es **87 mögliche Zeichen**.

Deshalb passieren am Ende noch zwei Schritte:

1. Eine letzte **LayerNorm** bringt die 96 Zahlen noch einmal auf eine stabile Größenordnung.
2. Eine **lineare Schicht** übersetzt die 96 Zahlen in **87 Zahlen** — eine für jedes mögliche nächste Zeichen.

```text
96 Zahlen
   ↓
LayerNorm
   ↓
Lineare Schicht
   ↓
87 Zahlen
```

Diese 87 Zahlen heißen **Logits**.

Man kann sie sich zunächst einfach als **Punktzahlen** vorstellen:

```text
"a"  →  1.2
"b"  → -0.7
"c"  →  3.8
"d"  →  0.4
...
```

Je größer der Logit, desto stärker spricht das Modell gerade für dieses Zeichen.

Aber diese Zahlen sind noch **keine Wahrscheinlichkeiten**. Sie können negativ sein und ergeben zusammen nicht 1.

Deshalb kommt **Softmax**.

Softmax verwandelt die 87 Punktzahlen in Wahrscheinlichkeiten:

```text
"a"  → 10 %
"b"  →  2 %
"c"  → 65 %
"d"  →  5 %
...
```

Danach gilt:

* jede Wahrscheinlichkeit liegt zwischen **0 und 1**
* alle 87 Wahrscheinlichkeiten ergeben zusammen **100 %**
* das Modell kann nun auswählen, welches Zeichen als Nächstes erzeugt wird

Mathematisch macht Softmax:

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Die Formel ist weniger wichtig als die Idee:

**Logits sind Punktzahlen. Softmax macht daraus Wahrscheinlichkeiten.**

▶️ Setzen wir jetzt die letzten Schritte zusammen:

In [ ]:
# ▶️ Abschluss-Norm und Ausgabeschicht
norm_ende = nn.LayerNorm(N_EMBD)
ausgabe = nn.Linear(N_EMBD, vokabular_groesse)

with torch.no_grad():
    letzter = bloecke(probe_x)              # (1, T, 96)
    logits = ausgabe(norm_ende(letzter))    # (1, T, 87)

print(f"nach dem letzten Block:  {tuple(letzter.shape)}")
print(f"nach der Ausgabeschicht: {tuple(logits.shape)}   (Batch, Länge, Vokabular)")
print()
print(f"Die Logits an der letzten Position, erste 8 von {vokabular_groesse}:")
print(logits[0, -1, :8].numpy().round(2))

In [ ]:
# ▶️ Die Logits der letzten Position als Wahrscheinlichkeiten
p = F.softmax(logits[0, -1], dim=-1)

plt.figure(figsize=(10, 4))
plt.bar(range(vokabular_groesse), p.numpy() * 100, color=BLAU, width=0.8)
plt.axhline(100 / vokabular_groesse, color=ORANGE, lw=2, ls="--",
            label=f"gleichverteilt: 1/{vokabular_groesse} = {100 / vokabular_groesse:.2f} %"
                  .replace(".", ","))
plt.xlabel("Zeichen (Nummer im Vokabular)")
plt.ylabel("Wahrscheinlichkeit in %")
plt.title(f"Nächstes Zeichen nach {PROBE_TEXT!r}\nuntrainiertes Modell")
plt.legend()
plt.show()

print(f"Summe aller Wahrscheinlichkeiten: {p.sum():.4f}")
print(f"kleinste {p.min() * 100:.2f} %   größte {p.max() * 100:.2f} %   "
      f"gleichverteilt wären {100 / vokabular_groesse:.2f} %")
print()
print("Die fünf wahrscheinlichsten Zeichen:")
for wert, nummer in zip(*torch.topk(p, 5)):
    print(f"  {zahl_zu_zeichen[nummer.item()]!r:>6}   {wert * 100:5.2f} %")

📖 Die Balken streuen um die gestrichelte Linie bei 1,15 % = 1/87. Der Mittelwert liegt exakt
dort — er muss es, weil sich 87 Wahrscheinlichkeiten zu 1 addieren. Die Streuung nach oben und
unten kommt allein aus der zufälligen Initialisierung der Ausgabeschicht: Die Logits sind nicht
alle gleich, also sind es auch die Wahrscheinlichkeiten nicht.

Wie inhaltsleer diese Reihenfolge ist, zeigt die Spitzengruppe: Dort stehen Zeichen, die nach
`Philosophie` nichts zu suchen haben. Plausibel wären ein Komma oder ein Zeilenumbruch — davon
weiß das Modell nichts. Es hat noch keinen Trainingsschritt gesehen.

---
## 5 · Das ganze Modell

📖 Alle Teile sind da. `MiniGPT` fasst sie zu einer Klasse zusammen:

1. Token-Vektor und Positions-Vektor nachschlagen und addieren
2. durch `N_LAYER` Blöcke schicken
3. abschließend normalisieren
4. mit der linearen Schicht auf `vokabular_groesse` Logits bringen

Der Rückgabewert ist ein Paar: die Logits und — falls Zielwerte übergeben werden — eine einzelne
Zahl, die misst, wie falsch das Modell liegt. Diese Zahl braucht erst das Training in Teil 4; die
Zeile steht in der Vorlage schon fertig drin.

### 🛠️ Challenge 4 — MiniGPT

Der Aufbau in `__init__` steht. Du schreibst den **Vorwärtsdurchlauf**: vier Zeilen, die die
Bausteine in der richtigen Reihenfolge anwenden.

*Tipp zu 1: Die Positionen sind `torch.arange(T, device=idx.device)` — `T` und nicht `KONTEXT`,
sonst passen die Formen bei kürzeren Texten nicht. Das `device=` sorgt dafür, dass es auch auf
einer Grafikkarte läuft.*

*Tipp zu 2: `self.bloecke` ist ein `nn.Sequential` und lässt sich wie eine einzelne Schicht
aufrufen.*

In [ ]:
class MiniGPT(nn.Module):
    """Ein vollständiges Decoder-Only-Sprachmodell."""

    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vokabular_groesse, N_EMBD)
        self.positions_embedding = nn.Embedding(KONTEXT, N_EMBD)
        self.bloecke = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.norm_ende = nn.LayerNorm(N_EMBD)
        self.ausgabe = nn.Linear(N_EMBD, vokabular_groesse)

    def forward(self, idx, ziele=None):
        B, T = idx.shape

        # TODO 1: Token-Vektoren und Positions-Vektoren addieren
        x = ...

        # TODO 2: durch alle Blöcke
        x = ...

        # TODO 3: Abschluss-Normalisierung
        x = ...

        # TODO 4: auf vokabular_groesse Logits, Form (B, T, 87)
        logits = ...

        if ziele is None:
            return logits, None

        # Diese Zeile gehört zum Training und ist Thema von Teil 4.
        verlust = F.cross_entropy(logits.view(B * T, vokabular_groesse), ziele.view(B * T))
        return logits, verlust

In [ ]:
# ✅ Selbsttest
torch.manual_seed(1337)
modell = MiniGPT().to(GERAET)
modell.eval()                                     # kein Dropout, damit die Prüfung eindeutig ist

x, y = hole_batch("train")

logits, verlust = modell(x)
assert isinstance(logits, torch.Tensor), \
    "forward() gibt keine Logits zurück — die TODOs sind noch offen"
assert logits.shape == (BATCH, KONTEXT, vokabular_groesse), \
    f"Erwartet {(BATCH, KONTEXT, vokabular_groesse)}, bekommen {tuple(logits.shape)}"
assert verlust is None, "Ohne Zielwerte muss der zweite Rückgabewert None sein"

logits, verlust = modell(x, y)
assert verlust is not None and verlust.ndim == 0, \
    "Mit Zielwerten muss eine einzelne Zahl zurückkommen"
assert torch.isfinite(verlust), "Der zweite Rückgabewert ist keine endliche Zahl"

kurz = torch.tensor([kodiere("Faust")], device=GERAET)
assert modell(kurz)[0].shape == (1, 5, vokabular_groesse), \
    "Auch kürzere Texte müssen gehen — torch.arange(T) statt torch.arange(KONTEXT)"

geaendert = x.clone()
geaendert[:, -1] = (x[:, -1] + 1) % vokabular_groesse
assert torch.allclose(modell(x)[0][:, :-1], modell(geaendert)[0][:, :-1], atol=1e-4), \
    "Eine Änderung am letzten Zeichen darf die Vorhersagen davor nicht verändern"

print("✅ Challenge 4 gelöst")
print()
print(f"Eingabe:  {tuple(x.shape)}   ← {BATCH} Textstücke à {KONTEXT} Zeichen")
print(f"Ausgabe:  {tuple(logits.shape)}   ← je Position {vokabular_groesse} Logits")

<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
    def forward(self, idx, ziele=None):
        B, T = idx.shape

        x = self.token_embedding(idx) + self.positions_embedding(torch.arange(T, device=idx.device))

        x = self.bloecke(x)

        x = self.norm_ende(x)

        logits = self.ausgabe(x)

        if ziele is None:
            return logits, None

        verlust = F.cross_entropy(logits.view(B * T, vokabular_groesse), ziele.view(B * T))
        return logits, verlust
```

Das Modell sagt an **jeder** Position ein nächstes Zeichen voraus, nicht nur am Ende. Die Ausgabe
`(32, 96, 87)` enthält 32 × 96 = 3.072 Vorhersagen aus einem einzigen Durchlauf. Das ist der
Grund, warum sich ein Sprachmodell effizient trainieren lässt.

</details>

▶️ Und jetzt die Frage, um die es in diesem Abschnitt geht: **Wo stecken die Parameter?**

In [ ]:
# ▶️ Parameter des Modells, aufgeschlüsselt
anzahl_parameter = sum(p.numel() for p in modell.parameters())

print(f"MiniGPT: {zahl(anzahl_parameter)} Parameter")
print()
for name, teil in [("Token-Embedding", modell.token_embedding),
                   ("Positions-Embedding", modell.positions_embedding),
                   ("Blöcke", modell.bloecke),
                   ("Abschluss-LayerNorm", modell.norm_ende),
                   ("Ausgabeschicht", modell.ausgabe)]:
    n = sum(p.numel() for p in teil.parameters())
    anteil = f"{n / anzahl_parameter:.1%}".replace(".", ",")
    print(f"  {name:<22}{zahl(n):>9}   {anteil:>6}")

In [ ]:
# ▶️ Innerhalb der Blöcke: Aufmerksamkeit oder Feed-Forward?
def zaehle(teile):
    """Parameter mehrerer Bausteine zusammenzählen."""
    return sum(p.numel() for teil in teile for p in teil.parameters())


aufmerksamkeit = zaehle([b.aufmerksamkeit for b in modell.bloecke])
feedforward = zaehle([b.feedforward for b in modell.bloecke])
normen = zaehle([b.norm1 for b in modell.bloecke] + [b.norm2 for b in modell.bloecke])
bloecke_gesamt = aufmerksamkeit + feedforward + normen

for name, n in [("Aufmerksamkeit", aufmerksamkeit),
                ("Feed-Forward", feedforward),
                ("LayerNorm", normen)]:
    anteil = f"{n / bloecke_gesamt:.1%}".replace(".", ",")
    print(f"  {name:<18}{zahl(n):>9}   {anteil:>6} der Blöcke")
print(f"  {'zusammen':<18}{zahl(bloecke_gesamt):>9}")
print()
print(f"Faktor Feed-Forward zu Aufmerksamkeit: {feedforward / aufmerksamkeit:.2f}"
      .replace(".", ","))

📖 **Zwei Drittel der Parameter eines Blocks stecken im Feed-Forward, ein Drittel in der
Aufmerksamkeit.** Das ist keine Eigenheit unseres kleinen Modells, sondern folgt direkt aus den
Formaten:

| | Matrizen | Parameter |
|---|---|---|
| Aufmerksamkeit | Query, Key, Value, Projektion | 4 · d² |
| Feed-Forward | d → 4d und 4d → d | 8 · d² |

Bei jeder Vektorlänge d dasselbe Verhältnis 1 : 2. Die Aufmerksamkeit bekommt in Vorträgen und
Abbildungen den meisten Platz; die Parameter liegen woanders.

---
## 6 · Was hast du gelernt?

📖 Das Modell ist vollständig. Ein Textstück geht hinein, 87 Wahrscheinlichkeiten kommen heraus,
alle Formen stimmen, die Maske hält.

Und es kann nichts. Die 360.855 Parameter stehen so, wie der Zufallsgenerator sie hingeschrieben
hat; das Modell hat keinen einzigen Trainingsschritt gesehen. Deshalb streuen die
Wahrscheinlichkeiten aus Abschnitt 4 zufällig um 1/87, statt ein Zeichen deutlich zu bevorzugen.

Was fehlt, ist die Kostenfunktion, der Optimierer und die Trainingsschleife.

---
### 🔬 Zum Weiterprobieren

Die Zelle unten rechnet die Parameterzahl für beliebige Einstellungen aus, ohne dass ein Modell
gebaut werden muss. Alles hier darf geändert werden.

In [ ]:
# ▶️ Spielfeld
MEIN_N_LAYER = 3
MEIN_N_EMBD = 96
MEIN_N_HEAD = 4


def parameter_zahl(n_layer, n_embd, n_head):
    """Die Aufschlüsselung für unsere Bauart, ohne dass ein Modell gebaut wird."""
    kopf_groesse = n_embd // n_head
    je_block = (n_head * 3 * n_embd * kopf_groesse + n_embd * n_embd + n_embd   # Aufmerksamkeit
                + 2 * n_embd * 4 * n_embd + 4 * n_embd + n_embd                 # Feed-Forward
                + 4 * n_embd)                                                   # zwei LayerNorm
    return {
        "Token-Embedding": vokabular_groesse * n_embd,
        "Positions-Embedding": KONTEXT * n_embd,
        "Blöcke": n_layer * je_block,
        "Abschluss-LayerNorm": 2 * n_embd,
        "Ausgabeschicht": n_embd * vokabular_groesse + vokabular_groesse,
    }


teile = parameter_zahl(MEIN_N_LAYER, MEIN_N_EMBD, MEIN_N_HEAD)
for name, n in teile.items():
    print(f"  {name:<22}{zahl(n):>10}")
print(f"  {'gesamt':<22}{zahl(sum(teile.values())):>10}")

**Drei Ideen:**

1. **Skip Connection entfernen.** Schreibe in `Block.forward` statt
   `x = x + self.aufmerksamkeit(self.norm1(x))` nur `x = self.aufmerksamkeit(self.norm1(x))`
   und führe den Selbsttest von Challenge 3 noch einmal aus. Welche Prüfung schlägt fehl?

2. **Post-Norm bauen.** Ändere beide Zeilen in `Block.forward` zur älteren Anordnung, also
   `x = self.norm1(x + self.aufmerksamkeit(x))` und `x = self.norm2(x + self.feedforward(x))`.
   Genau eine Prüfung des Selbsttests schlägt dann fehl — welche, und warum?

3. **An den Einstellungen drehen.** Setze im Spielfeld `MEIN_N_EMBD` auf 192 und `MEIN_N_LAYER`
   auf 6. Wie ändert sich die Parameterzahl, und welcher Posten wächst am stärksten? Danach
   `MEIN_N_HEAD` auf 8 — warum ändert sich dabei nichts?